# MiniReasoner: Model Evaluation & Benchmarking

This notebook evaluates trained **MiniReasoner RoPE** checkpoints downloaded from **Azure Blob Storage** (`minireasoner`) or loaded from a local `.pt` file.

### Key Capabilities:
- **Device-Agnostic Support**: Runs seamlessly on **CPU** (local laptops/machines in FP32) and **CUDA GPU** (e.g. NVIDIA A30 in BF16 or FP32).
- **Automatic Run Discovery & Selection**: Discovers available training runs under `checkpoints/Shared/MiniReasoner/` in Azure Blob Storage, selects the newest run (or an explicit `RUN_ID`), ranks checkpoints by validation loss, and caches them per `run_id`.
- **RoPE Architecture & Weight Inspection**: Verifies the 32.05M-parameter RoPE transformer configuration (`context_length = 1,024`), positional encoding parameters, and embedded Byte-Level BPE tokenizer.
- **Pre-Training Convergence Plots**: Visualizes interval-mean training loss and validation loss across all logged steps (`.4f` precision).
- **Flexible Text Generation & Long-Context (>256 tokens) Verification**: Evaluates narrative generation across greedy decoding, nucleus sampling, and long prompts exercising the 1,024-token RoPE window.
- **Quantitative Perplexity, Arithmetic Baseline, & Throughput Benchmarks**: Measures cross-entropy / PPL, the pre-SFT arithmetic baseline from `POST_TRAINING_PLAN.md`, and tokens/second throughput.

## 1. Project Environment Setup
Load environment variables (including Azure Blob Storage connection strings), locate the repository root, and configure `sys.path`.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Locate MiniReasoner repository root
working_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / "src" / "train.py").is_file() and (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the MiniReasoner repository.")

# Load Azure credentials from .env
load_dotenv(project_root / ".env")
print(f"Project root located at: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## 2. Hardware & Device Configuration (CPU / GPU Compatibility)
Inference automatically detects whether a CUDA GPU is available (`DEVICE_PREFERENCE = "auto"`), or you can force `"cpu"` / `"cuda"`:
- **CUDA GPU**: Uses native BF16 on Ampere+ (A30, A100, RTX 30/40xx) or FP32 on older GPUs.
- **CPU**: Uses standard Float32 cleanly with no CUDA drivers required.

In [ ]:
import torch

# Device selection: 'auto' (preferred), 'cuda', or 'cpu'
DEVICE_PREFERENCE = "auto"

if DEVICE_PREFERENCE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
elif DEVICE_PREFERENCE in ("cuda", "cpu"):
    if DEVICE_PREFERENCE == "cuda" and not torch.cuda.is_available():
        print("Notice: CUDA requested but unavailable on this system. Falling back to CPU.")
        device = torch.device("cpu")
    else:
        device = torch.device(DEVICE_PREFERENCE)
else:
    raise ValueError(f"Unknown device preference: {DEVICE_PREFERENCE}")

print("=" * 60)
print(f"ACTIVE EXECUTION DEVICE: {device.type.upper()}")
print("=" * 60)

if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_cap = torch.cuda.get_device_capability(0)
    bf16_supported = torch.cuda.is_bf16_supported() and cuda_cap[0] >= 8
    dtype = torch.bfloat16 if bf16_supported else torch.float32
    print(f"GPU Hardware:       {gpu_name}")
    print(f"Total VRAM:         {vram_gb:.2f} GB")
    print(f"Compute Capability: {cuda_cap[0]}.{cuda_cap[1]}")
    print(f"Inference Dtype:    {dtype}")
else:
    dtype = torch.float32
    num_threads = torch.get_num_threads()
    print(f"CPU Threads:        {num_threads}")
    print("Inference Dtype:    torch.float32")
    print("Mode:               Portable CPU Inference")
print("=" * 60)

## 3. Discover & Retrieve Best Model from Azure Blob Storage
MiniReasoner training checkpoints are saved under `checkpoints/Shared/MiniReasoner/<run_id>/` in Azure Blob Storage.

- Leave `RUN_ID = None` to automatically discover runs in Azure and pick the most recently updated run.
- Or set `RUN_ID = "<mlflow_run_id>"` to inspect a specific run.
- Set `LOCAL_CHECKPOINT = "path/to/file.pt"` for completely offline evaluation.
- Downloaded checkpoints are cached cleanly under `checkpoints/<run_id>/` so different runs never overwrite each other.

In [ ]:
import re
from src.azure_blob import container_client
from src.checkpoint_store import CheckpointStore, _FILENAME
from src.settings import AZURE_CONTAINER, MLFLOW_EXPERIMENT

# Configuration overrides
RUN_ID = None              # Set to an MLflow run ID, or None to auto-select the newest run in Azure
CHECKPOINT_ALIAS = "best"  # 'best' (lowest val loss), 'latest' (highest step), or 'step-N'
LOCAL_CHECKPOINT = None    # Optional path to a local .pt file for offline evaluation

checkpoints_dir = project_root / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)
best_snapshot = None

if LOCAL_CHECKPOINT is not None:
    cached_checkpoint_path = Path(LOCAL_CHECKPOINT).expanduser().resolve()
    if not cached_checkpoint_path.is_file():
        raise FileNotFoundError(f"Local checkpoint not found: {cached_checkpoint_path}")
    run_id = cached_checkpoint_path.parent.name
    print(f"Using offline local checkpoint: {cached_checkpoint_path}")
else:
    folder = MLFLOW_EXPERIMENT.strip("/")
    exp_prefix = f"checkpoints/{folder}/"
    container = container_client(AZURE_CONTAINER)

    if RUN_ID is None:
        runs_info = {}
        run_pattern = re.compile(rf"^{re.escape(exp_prefix)}([A-Za-z0-9_-]+)/([^/]+)$")
        for blob in container.list_blobs(name_starts_with=exp_prefix):
            match = run_pattern.match(blob.name)
            if not match:
                continue
            rid, fname = match.groups()
            if not _FILENAME.fullmatch(fname):
                continue
            entry = runs_info.setdefault(rid, {"count": 0, "last_modified": blob.last_modified})
            entry["count"] += 1
            if blob.last_modified and (entry["last_modified"] is None or blob.last_modified > entry["last_modified"]):
                entry["last_modified"] = blob.last_modified

        if not runs_info:
            raise FileNotFoundError(f"No checkpoints found under {exp_prefix} in Azure container '{AZURE_CONTAINER}'.")

        discovered = sorted(
            runs_info.items(),
            key=lambda kv: kv[1]["last_modified"].timestamp() if kv[1]["last_modified"] else 0.0,
            reverse=True,
        )
        print(f"Discovered {len(discovered)} run(s) in Azure container '{AZURE_CONTAINER}':\n")
        print(f"{'Run ID':<36}{'Snapshots':<12}{'Last Updated (UTC)'}")
        print("-" * 72)
        for rid, info in discovered:
            ts = info["last_modified"].strftime("%Y-%m-%d %H:%M:%S") if info["last_modified"] else "N/A"
            print(f"{rid:<36}{info['count']:<12}{ts}")
        run_id = discovered[0][0]
        print(f"\nAuto-selected newest run: {run_id}")
    else:
        run_id = RUN_ID

    store = CheckpointStore(AZURE_CONTAINER, run_id)
    versions = store._versions()
    if not versions:
        raise FileNotFoundError(f"No valid checkpoints found for run {run_id!r} in {store.location}.")

    print(f"\nDiscovered {len(versions)} checkpoint snapshot(s) for run {run_id}:\n")
    print(f"{'Rank':<6}{'Step':<8}{'Val Loss':<12}{'Train Mean':<12}{'Filename'}")
    print("-" * 85)
    for i, snap in enumerate(versions[:5], start=1):
        print(f"#{i:<5}{snap.step:<8}{snap.val_loss:<12.4f}{snap.train_loss_interval_mean:<12.4f}{snap.name[:45]}...")

    if CHECKPOINT_ALIAS == "best":
        target_snapshot = versions[0]
    elif CHECKPOINT_ALIAS == "latest":
        target_snapshot = max(versions, key=lambda item: item.step)
    elif alias_match := re.fullmatch(r"step-(\d+)", CHECKPOINT_ALIAS):
        step_num = int(alias_match[1])
        target_snapshot = next((item for item in versions if item.step == step_num), None)
        if target_snapshot is None:
            raise FileNotFoundError(f"Step {step_num} not found in run {run_id}.")
    else:
        raise ValueError("CHECKPOINT_ALIAS must be 'best', 'latest', or 'step-N'.")

    best_snapshot = target_snapshot
    print(f"\n--> Selected ({CHECKPOINT_ALIAS}): Step {target_snapshot.step:,} | Val Loss: {target_snapshot.val_loss:.4f} | Train Mean: {target_snapshot.train_loss_interval_mean:.4f}")

    # Run-scoped local cache so checkpoints from different runs never collide
    run_cache_dir = checkpoints_dir / run_id
    run_cache_dir.mkdir(parents=True, exist_ok=True)
    cached_checkpoint_path = run_cache_dir / f"{CHECKPOINT_ALIAS}_step_{target_snapshot.step}_{target_snapshot.sha256[:8]}.pt"

    if not cached_checkpoint_path.is_file():
        print(f"Downloading checkpoint from Azure to {cached_checkpoint_path}...")
        store.download(CHECKPOINT_ALIAS, cached_checkpoint_path)
        print("Download complete and verified via SHA256 checksum!")
    else:
        size_mb = cached_checkpoint_path.stat().st_size / (1024 * 1024)
        print(f"Using locally cached checkpoint: {cached_checkpoint_path} ({size_mb:.2f} MB)")

## 4. Load Checkpoint and Inspect Model Architecture
We load the checkpoint dictionary into memory:
- **Weights & RoPE Buffers**: Transferred cleanly to the active device (`device`) and dtype (`dtype`).
- **Tokenizer**: Reconstructed from the embedded `tokenizer_json` (Byte-Level BPE with 1,024 tokens).
- **Configuration**: `GPTConfig` dimensions (10 layers, 8 heads, 512 embedding dimension, `context_length = 1,024`, RoPE $\theta = 10,000$).

In [ ]:
from tokenizers import Tokenizer
from src.model import GPT, GPTConfig
from src.train import load_checkpoint
from src.tokenizer import EOS

print(f"Loading checkpoint weights from: {cached_checkpoint_path}")
checkpoint = load_checkpoint(cached_checkpoint_path)

if "transformer.wpe.weight" in checkpoint.get("model", {}):
    raise RuntimeError(
        f"Checkpoint {cached_checkpoint_path.name} uses legacy learned positional embeddings (wpe). "
        "Select a RoPE training run via RUN_ID or run a new RoPE training job first."
    )

# Initialize tokenizer and RoPE GPTConfig from checkpoint
tokenizer = Tokenizer.from_str(checkpoint["tokenizer_json"])
model_config = GPTConfig(**checkpoint["model_config"])

model = GPT(model_config).to(device).eval()
model.load_state_dict(checkpoint["model"])
if device.type == "cuda" and dtype == torch.bfloat16:
    model = model.to(dtype=torch.bfloat16)

# Inspect architecture & parameter counts dynamically
total_params = sum(p.numel() for p in model.parameters())
wte_params = model.transformer.wte.weight.numel()
blocks_params = sum(p.numel() for p in model.transformer.h.parameters())
ln_f_params = sum(p.numel() for p in model.transformer.ln_f.parameters())
head_dim = model_config.n_embd // model_config.n_head
pos_desc = f"RoPE (theta={model_config.rope_theta:g}, head_dim={head_dim})" if model_config.pos_encoding == "rope" else model_config.pos_encoding

print("\n" + "=" * 58)
print("MINIREASONER MODEL ARCHITECTURE")
print("=" * 58)
print(f"Positional Encoding:        {pos_desc}")
print(f"Transformer Layers (Blocks): {model_config.n_layer}")
print(f"Attention Heads:            {model_config.n_head} ({head_dim} dim/head)")
print(f"Embedding Dimension:        {model_config.n_embd}")
print(f"Context Window Length:      {model_config.context_length:,} tokens")
print(f"Vocabulary Size:            {model_config.vocab_size:,} tokens")
print(f"Total Parameters:           {total_params:,} (~{total_params / 1e6:.2f}M)")
print(f"Token Embeddings (tied):    {wte_params:,}")
print(f"Transformer Core (Blocks):  {blocks_params:,}")
print(f"Final LayerNorm:            {ln_f_params:,}")
print(f"EOS Token:                  '{EOS}' (ID: {tokenizer.token_to_id(EOS)})")
print(f"Checkpoint Training Step:   {checkpoint.get('step', best_snapshot.step if best_snapshot else 0):,}")
print(f"Active Device / Dtype:      {device.type.upper()} / {dtype}")
print("=" * 58)

## 5. Pre-Training Loss Curve Analysis
The checkpoint stores the historical metrics logged throughout pre-training. We plot the validation loss and interval-mean training loss across all recorded steps.

In [ ]:
import matplotlib.pyplot as plt

metrics_history = checkpoint.get("metrics", [])

if metrics_history:
    val_points = [(m["step"], m["val_loss"]) for m in metrics_history if m.get("val_loss") is not None]
    train_points = [(m["step"], m["train_loss_interval_mean"]) for m in metrics_history if m.get("train_loss_interval_mean") is not None]

    plt.figure(figsize=(10, 5), dpi=100)
    if train_points:
        plt.plot([s for s, _ in train_points], [v for _, v in train_points], label="Train Loss (Interval Mean)", color="#2563eb", linewidth=2)
    if val_points:
        plt.plot([s for s, _ in val_points], [v for _, v in val_points], label="Validation Loss", color="#dc2626", linewidth=2, linestyle="--")

    max_step = max((m["step"] for m in metrics_history), default=0)
    plt.title(f"MiniReasoner Pre-Training Convergence ({max_step:,} Steps | Run {run_id[:8]})", fontsize=13, fontweight="bold")
    plt.xlabel("Optimizer Step", fontsize=11)
    plt.ylabel("Cross-Entropy Loss", fontsize=11)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    if val_points:
        print(f"Initial Validation Loss: {val_points[0][1]:.4f} (Step {val_points[0][0]:,})")
        print(f"Final Validation Loss:   {val_points[-1][1]:.4f} (Step {val_points[-1][0]:,})")
        best_val_step, best_val_loss = min(val_points, key=lambda x: x[1])
        print(f"Best Validation Loss:    {best_val_loss:.4f} (Step {best_val_step:,})")
        print(f"Net Loss Improvement:    {val_points[0][1] - best_val_loss:.4f}")
elif best_snapshot is not None:
    print(f"Step {best_snapshot.step:,}: Validation Loss = {best_snapshot.val_loss:.4f}, Train Mean = {best_snapshot.train_loss_interval_mean:.4f}")

## 6. Device-Agnostic Generation Engine
Inference works identically on **both CPU (FP32) and CUDA GPU (BF16/FP32)** with FP32 logit sampling for numerical stability and automatic cropping to `model.config.context_length` (`1,024`).

In [ ]:
import time
from typing import Dict, Any

@torch.inference_mode()
def generate(
    model: torch.nn.Module,
    tokenizer: Tokenizer,
    prompt: str,
    max_new_tokens: int = 150,
    temperature: float = 0.0,
    top_k: int = 50,
    top_p: float = 0.9,
    repetition_penalty: float = 1.0,
    device: torch.device = device,
) -> Dict[str, Any]:
    model.eval()
    eos_id = tokenizer.token_to_id(EOS)
    context_length = model.config.context_length

    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    if not ids:
        ids = [eos_id]
    if len(ids) > context_length:
        ids = ids[-context_length:]

    generated_tokens = []
    start_time = time.perf_counter()

    for _ in range(max_new_tokens):
        input_window = ids[-context_length:]
        x = torch.tensor([input_window], dtype=torch.long, device=device)

        # Compute sampling logits in FP32 so BF16 GPU and FP32 CPU behave consistently
        logits = model(x)[0, -1].float()

        if repetition_penalty != 1.0:
            for prev_token in set(input_window):
                if logits[prev_token] < 0:
                    logits[prev_token] *= repetition_penalty
                else:
                    logits[prev_token] /= repetition_penalty

        if temperature <= 0.0 or top_k == 1:
            next_token_id = torch.argmax(logits).item()
        else:
            scaled_logits = logits / temperature

            if top_k > 0:
                v, _ = torch.topk(scaled_logits, min(top_k, scaled_logits.size(-1)))
                scaled_logits[scaled_logits < v[-1]] = -float("Inf")

            if 0.0 < top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)
                cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                scaled_logits[indices_to_remove] = -float("Inf")

            probs = torch.softmax(scaled_logits, dim=-1)
            next_token_id = torch.multinomial(probs, num_samples=1).item()

        if next_token_id == eos_id:
            break

        ids.append(next_token_id)
        generated_tokens.append(next_token_id)

    elapsed_time = time.perf_counter() - start_time
    completion_text = tokenizer.decode(generated_tokens)
    tokens_per_sec = len(generated_tokens) / elapsed_time if elapsed_time > 0 else 0.0

    return {
        "prompt": prompt,
        "completion": completion_text,
        "full_text": prompt + completion_text,
        "tokens_generated": len(generated_tokens),
        "elapsed_seconds": elapsed_time,
        "tokens_per_second": tokens_per_sec,
        "device": device.type,
    }

print("Device-agnostic generator ready.")

## 7. Qualitative Evaluation: Story Completion Benchmarks
We test deterministic greedy decoding (`temperature = 0.0`), stochastic nucleus sampling (`temperature = 0.7`, `top_p = 0.9`), and a long-context prompt (>256 tokens) exercising the 1,024 RoPE context window.

In [ ]:
story_prompts = [
    "Once upon a time, there was a little",
    "One day, Lily found a secret",
    "Mia had a little dog named",
    "Jack and his best friend went to the",
    "In the middle of the dark forest,",
]

print("=" * 70)
print(f"STORY COMPLETION BENCHMARK: GREEDY DECODING (Running on {device.type.upper()})")
print("=" * 70)

for idx, p in enumerate(story_prompts, start=1):
    result = generate(model, tokenizer, p, max_new_tokens=100, temperature=0.0, device=device)
    print(f"\n--- [Story Prompt {idx}]: \"{p}\" ---")
    print(f"Speed: {result['tokens_per_second']:.1f} tok/s | Tokens: {result['tokens_generated']} | Time: {result['elapsed_seconds']:.2f}s")
    print(result["full_text"].strip())

In [ ]:
print("=" * 70)
print(f"STORY COMPLETION BENCHMARK: NUCLEUS SAMPLING (T=0.7, top_p=0.9 on {device.type.upper()})")
print("=" * 70)

for idx, p in enumerate(story_prompts[:3], start=1):
    result = generate(model, tokenizer, p, max_new_tokens=100, temperature=0.7, top_p=0.9, device=device)
    print(f"\n--- [Sampling Variant {idx}]: \"{p}\" ---")
    print(f"Speed: {result['tokens_per_second']:.1f} tok/s | Tokens: {result['tokens_generated']} | Time: {result['elapsed_seconds']:.2f}s")
    print(result["full_text"].strip())

In [ ]:
# Long-context test (>256 tokens) verifying 1,024 RoPE context handling
long_prompt = (
    "Once upon a time, in a quiet village beside a silver river, there lived a young clockmaker named Clara. "
    "Every morning she swept the wooden floor of her shop, polished the brass gears on the workbench, and checked the tall grandfather clock by the window. "
    "The villagers loved visiting Clara because she could repair anything from tiny pocket watches to musical jewelry boxes. "
    "One autumn afternoon, an old traveler in a heavy wool coat stepped inside carrying a locked lantern made of dark iron and blue glass. "
    "He placed the lantern gently on the counter and told Clara that it had belonged to the lighthouse keeper many years ago before the storm broke its chime. "
    "Clara examined the delicate copper springs inside, took out her smallest magnifying lens, and noticed an engraved map hidden beneath the brass dial. "
    "Together with her clever fox Pip, who always slept near the warm stove, Clara carefully turned the tiny silver screw at the base of the lantern. "
    "With a soft click, the blue glass door swung open and revealed a glowing compass pointing north toward the misty hills. "
    "Clara looked at Pip, smiled, and reached for her coat because she knew their adventure was about to begin. "
    "As they stepped outside onto the cobblestone street,"
)
long_prompt_tokens = len(tokenizer.encode(long_prompt, add_special_tokens=False).ids)
print("=" * 70)
print(f"LONG-CONTEXT ROPE CHECK ({long_prompt_tokens} prompt tokens on {device.type.upper()})")
print("=" * 70)
long_res = generate(model, tokenizer, long_prompt, max_new_tokens=120, temperature=0.0, device=device)
print(f"Generated {long_res['tokens_generated']} tokens at {long_res['tokens_per_second']:.1f} tok/s:")
print(long_res["completion"].strip())

## 8. Quantitative Evaluation: Perplexity on SimpleStories Passages
We evaluate cross-entropy loss (`.4f`) and Perplexity ($PPL = \exp(\mathcal{L})$) across both short passages and the long multi-sentence (>256 token) RoPE passage.

In [ ]:
import math
import numpy as np

reference_passages = [
    "Once upon a time, there was a little girl named Lily. She loved to play in the big garden with her red ball.",
    "One sunny afternoon, Tim and his puppy decided to visit the river. They saw small fish swimming in the clear water.",
    "A kind bird flew down from the big tree and chirped happily. It found a tiny bread crumb on the green grass.",
    "Mia had an old shiny key that opened the wooden chest. Inside the chest, there was a warm woolen sweater.",
    "The little boy wanted to help his mother bake a sweet apple pie. He carefully washed each apple.",
    long_prompt,
]

print("=" * 65)
print(f"{'Sample ID':<12}{'Tokens':<10}{'Cross-Entropy Loss':<22}{'Perplexity (PPL)'}")
print("=" * 65)

ce_losses = []
ppl_scores = []

for idx, passage in enumerate(reference_passages, start=1):
    token_ids = tokenizer.encode(passage, add_special_tokens=False).ids
    if len(token_ids) < 2:
        continue
    token_ids = token_ids[:model_config.context_length]
    x = torch.tensor([token_ids[:-1]], dtype=torch.long, device=device)
    y = torch.tensor([token_ids[1:]], dtype=torch.long, device=device)

    with torch.inference_mode():
        logits = model(x).float()
        ce = torch.nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten()).item()
        ppl = math.exp(ce)

    ce_losses.append(ce)
    ppl_scores.append(ppl)
    label = f"Passage #{idx}" if idx < len(reference_passages) else "Long (>256)"
    print(f"{label:<12}{len(token_ids):<10}{ce:<22.4f}{ppl:.2f}")

print("-" * 65)
print(f"Mean Cross-Entropy Loss: {np.mean(ce_losses):.4f}")
print(f"Mean Perplexity (PPL):   {np.mean(ppl_scores):.2f}")
print("=" * 65)

## 9. Arithmetic Baseline Evaluation (Pre-Training Baseline)
As defined in **Section 2 of `POST_TRAINING_PLAN.md`**, before initiating Supervised Fine-Tuning (SFT) or Reinforcement Learning (RL), we measure the pre-trained base model's baseline performance on 1-digit and 2-digit arithmetic problems.

Standard prompt format: `What is A + B?\nAnswer:`

In [ ]:
import re

arithmetic_benchmarks = [
    {"op": "add_1d", "a": 3, "b": 5, "answer": 8, "prompt": "What is 3 + 5?\nAnswer:"},
    {"op": "add_1d", "a": 7, "b": 8, "answer": 15, "prompt": "What is 7 + 8?\nAnswer:"},
    {"op": "add_2d", "a": 12, "b": 15, "answer": 27, "prompt": "What is 12 + 15?\nAnswer:"},
    {"op": "add_2d_carry", "a": 47, "b": 8, "answer": 55, "prompt": "What is 47 + 8?\nAnswer:"},
    {"op": "add_2d_carry", "a": 38, "b": 24, "answer": 62, "prompt": "What is 38 + 24?\nAnswer:"},
    {"op": "sub_1d", "a": 9, "b": 4, "answer": 5, "prompt": "What is 9 - 4?\nAnswer:"},
    {"op": "sub_2d", "a": 25, "b": 10, "answer": 15, "prompt": "What is 25 - 10?\nAnswer:"},
    {"op": "sub_2d_borrow", "a": 52, "b": 17, "answer": 35, "prompt": "What is 52 - 17?\nAnswer:"},
    {"op": "sub_2d_borrow", "a": 43, "b": 18, "answer": 25, "prompt": "What is 43 - 18?\nAnswer:"},
    {"op": "add_zero", "a": 14, "b": 0, "answer": 14, "prompt": "What is 14 + 0?\nAnswer:"},
]

print("=" * 85)
print("ARITHMETIC BASELINE EVALUATION (Pre-Trained Base Model)")
print("=" * 85)

results = []
for item in arithmetic_benchmarks:
    res = generate(model, tokenizer, item["prompt"], max_new_tokens=15, temperature=0.0, device=device)
    raw_output = res["completion"].strip()

    match = re.search(r"^-?\d+", raw_output)
    extracted = int(match.group(0)) if match else None

    exact_match = (extracted == item["answer"])
    valid_format = (match is not None)

    results.append({
        "prompt": item["prompt"].replace("\n", " "),
        "expected": item["answer"],
        "model_output": raw_output[:30].replace("\n", " "),
        "extracted": extracted,
        "exact_match": exact_match,
        "valid_format": valid_format,
        "op_category": item["op"]
    })

print(f"{'Prompt':<25}{'Target':<10}{'Raw Model Output':<30}{'Exact Match':<14}{'Valid Format'}")
print("-" * 85)
for r in results:
    print(f"{r['prompt']:<25}{r['expected']:<10}{r['model_output']:<30}{str(r['exact_match']):<14}{str(r['valid_format'])}")

exact_count = sum(1 for r in results if r["exact_match"])
format_count = sum(1 for r in results if r["valid_format"])
print("-" * 85)
print(f"Exact Match Accuracy: {exact_count}/{len(results)} ({exact_count / len(results) * 100:.1f}%)")
print(f"Valid Format Rate:    {format_count}/{len(results)} ({format_count / len(results) * 100:.1f}%)")

## 10. Hardware Performance & Throughput Benchmark
We benchmark generation throughput (tokens/second) and total latency across different completion lengths.

In [ ]:
token_lengths = [30, 60, 100]
bench_prompt = "Once upon a time, there was a little"

print("=" * 70)
print(f"THROUGHPUT BENCHMARK (Hardware: {device.type.upper()})")
print("=" * 70)
print(f"{'Target Tokens':<15}{'Tokens Generated':<20}{'Elapsed Time (s)':<20}{'Throughput (tok/s)'}")
print("-" * 70)

for target in token_lengths:
    bench_res = generate(model, tokenizer, bench_prompt, max_new_tokens=target, temperature=0.0, device=device)
    print(f"{target:<15}{bench_res['tokens_generated']:<20}{bench_res['elapsed_seconds']:<20.2f}{bench_res['tokens_per_second']:.2f}")

print("=" * 70)

## 11. Summary & Next Steps
- **Run Selection**: `RUN_ID = None` automatically picks the most recently updated run in Azure Blob Storage and caches its best checkpoint under `checkpoints/<run_id>/`.
- **CPU & GPU Portable**: Section 2 automatically picks CUDA (`bf16` on Ampere+ GPUs) when available and falls back cleanly to CPU (`float32`).
- **RoPE & 1,024 Context**: Section 4 verifies the RoPE architecture (`32,049,152` base parameters) and Section 7 + 8 verify coherence and perplexity on prompts longer than 256 tokens.
- **Next Phase**: Use the verified best pre-trained checkpoint and embedded tokenizer as the starting model for Phase 3 arithmetic Supervised Fine-Tuning (SFT) in `POST_TRAINING_PLAN.md`.